# Studio 7: Build a Reproducible First Analysis

**Topic 07 · 2 lectures**

<hr>

<center>
<div>
<img src="https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/images/edrai_logo.png" width="300"/>
</div>
</center>


# <center><a class="tocSkip"></center>
# <center>HONR 46400 — Evidence-Driven Research</center>
# <center>Professor: Davi Moreira</center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/notebooks/student/nb07_experimental_descriptive_student.ipynb)

---

## 🧭 Inquiry & Claim Boundary

**Inquiry emphasis:** all positions (the execution week). This week is about the
trip from words to running code, and the discipline is identical whatever your
compass position. A **pipeline** is the chain of code steps that carries raw data
to a reported result: load the data, transform it into the analysis table,
estimate the declared quantity. The question this week patrols is whether each
step of that chain executes the analysis you declared, and nothing else.

**Design pathway:** cross-cutting. Every route builds its pipeline this week. The
common demonstration uses a real field experiment; your own pipeline stays on the
route you declared.

| | |
|---|---|
| **A claim this topic PERMITS** | "My pipeline implements my declared analysis: for every code step I can name the Contract field it executes, and my first result carries its uncertainty statement, labeled provisional." |
| **A claim this topic does NOT permit** | "The notebook ran without errors, so the analysis is correct," or a polished point estimate reported with no uncertainty and no output it traces to. |

**Where this sits in the course:** Week 7, the build week of Studio 7. It stands
on your route declaration (M4) and your governed data and measurement record
(M5), and it develops **M6, your first executable analysis with its URC abstract
internal gate**, which you work on at this week's Friday studio. Next week the
same pipeline faces a clean restart (M7).

## Learning Objectives

By the end of this notebook, you will be able to:

1. Map every field of your **Research Contract** to the pipeline step that
   implements it, in a Contract-to-code map you can defend line by line.
2. Predict a pipeline's output before running it, then reconcile each executed
   step against the declared analysis instead of trusting a clean run.
3. Build and run a three-step pipeline (load, transform, estimate) on a real
   field experiment, producing one traceable first result.
4. Attach an **uncertainty statement** to a point estimate, and refuse to report
   a result without one.
5. Verify delegated code with a **known-answer test** and a **line review**, and
   produce the environment record and claim-output trace that make your result
   checkable by a stranger.
6. Apply all of it to your own project's **M6**: the first executable analysis,
   with a URC abstract held inside what the current result licenses.

---

In [ ]:
# Setup — run this cell first.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx  # for the pipeline map; if missing locally: pip install networkx (pre-installed in Colab)

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (9, 5)

SEED = 464  # course number — keeps every simulation reproducible
rng = np.random.default_rng(SEED)

# Data loads: GitHub raw URL first (works in Colab), local repo path as fallback.
DATA_URL = ("https://raw.githubusercontent.com/davi-moreira/"
            "2026F_evidence_driven_research_purdue_HONR464/main/notebooks/data/")

def load_course_data(filename):
    """Load a course dataset from GitHub, falling back to the local repo copy."""
    try:
        return pd.read_csv(DATA_URL + filename)
    except Exception:
        from pathlib import Path
        local = Path("notebooks/data") / filename
        if not local.exists():
            local = Path("../data") / filename
        return pd.read_csv(local)

print("✓ Setup complete — seed", SEED)

# Lecture 1

🗺️ **Today's frame (Monday, 9 / 22 / 12 / 7):** open with the 🧩 puzzle and 🔮 **Predict First** · investigate with 🛠️ **Run the Study** and your AI · verify with 🔍 **Read the Evidence** · drill 📝 **Practice** aloud and take ⚖️ **Make a Design Choice** · close in the room with 🎯 **Take It to Your Project**, 🛡️ **Defend Your Decision**, and your 📒 ledger row. All seven moves happen in class; 🏠-marked items and everything below the ⏸ line are optional depth.

### 🎤 SRL Lead Brief

*This lecture opens with its Student Research Lead. If this is your slot, this
brief is your backbone. If not, read along: this is how the room will run.*

**Your mission.** By minute 50 everyone in the room can map each field of a
Research Contract to the pipeline step that implements it, and can name the
first step where a cleanly running pipeline changed what it estimates.

**Run of show (Monday: 9 / 22 / 12 / 7).**

| Min | Section | What you do |
|---|---|---|
| 0–9 | Research puzzle | Present the colleague pipeline below and make everyone commit an answer in writing before any AI is opened. |
| 9–31 | Guided AI investigation | Steer the room through sections 1 and 2 with your AI: first the pipeline map, then the live prompt on the three-step build. Keep asking one thing of every answer it returns: which quantity does this code compute, and over which cases? |
| 31–43 | Verification + formalization | The room reads the step receipts off the printout: 8,375 rows kept, arms of 6,104 and 2,271, a gap of +3.41 points. Reconcile each step against the demonstration Contract, then hand the verified map to your instructor for the formal wrap on pipelines. |
| 43–50 | Decision and defense | Everyone records the decision and a ledger entry; close on a spoken Claim Ticket. |

**Three questions that keep the room thinking.**

1. *"What quantity does this code compute, and over which cases?"* Listen for:
   the declared quantity named in words, plus the exact set of rows. Watch out
   for: someone quoting the printed number as if the number were the answer.
2. *"The code ran with no errors. What does that prove?"* Listen for: only that
   it executed. Correctness is checked against the declaration, not against the
   console.
3. *"Where in the colleague's four steps did the estimand change?"* Listen for:
   the filter in step 2, which shrank the case set. Watch out for: blaming the
   estimate line, which faithfully computed the wrong table's number.

**One AI trap to watch for.** Someone will paste code into a chatbot and ask "is
this correct?" The reply praises tidy code: clear names, sensible structure,
no bugs it can see. It is reviewing style, because it has never read the
Contract this code is supposed to implement. When it appears, read the praise
aloud, then make the room paste the declared quantity next to the code and ask
the question again. Watch the verdict change.

**Checkpoints.** By minute 9, everyone has written yes or no on whether the
colleague's pipeline estimates its declared quantity, with a step number. By
minute 31, the three-step printout is up and each step is annotated with the
Contract field it implements. By minute 43, everyone has drafted the first line
of their own pipeline map, which is where M6 starts.

**Make it yours.** The frame, the puzzle, and the checkpoints are fixed; the
staging is yours. Swap the colleague case for a pipeline from your own field, a
sensor feed that silently drops offline hours, a survey pipeline that deletes
non-answers, a finance script that keeps only complete quarters; or stage the
puzzle as a courtroom where the code is the witness and the Contract is the
testimony it must match. In your prep script, name one thing you are adding
that this brief does not contain.

**Prep.** Start about a week ahead. Submit your preparation script or notebook
two days ahead. The full guide is the Student Research Lead handbook in the
course materials.

### 🧩 Research Puzzle

*(Your research lead opens the lecture with this. Think it through and commit an
answer before we go further. No AI yet.)*

Your instructor issues this fragment from a colleague's dossier.

> **SIMULATED CASE · TRAINING MATERIAL · NOT EVIDENCE**
>
> *Declared analysis (from the colleague's Contract):* "the difference in 2014
> turnout between the contacted arm and the control arm, over all 8,375 voters
> in the experiment file, in percentage points."
>
> *The pipeline (runs top to bottom with no errors):*
>
> ```python
> df = load("foos_etal.csv")                                     # step 1
> df = df[df["ward"] == "F"]                                     # step 2: focus where canvassing was most active
> arm_means = df.groupby("treat")["marked_register_2014"].mean() # step 3
> print(arm_means[1] - arm_means[0])                             # step 4: prints -0.1019
> ```

The code executed cleanly and printed a real number. Here is the question on the
table: **does this pipeline estimate the quantity the Contract declared?**
Commit yes or no in writing. If no, name the exact step where the estimand
changed, and say what the printed number is actually a number *of*. Holding a
green-running pipeline against its own declaration is the whole job of
Lecture 1.

## 1. From the Contract to a Code Map

**Guiding question:** *does each code step execute the analysis you declared?*

> *"I do not review the number first. I review the path. Show me, for every step
> in your pipeline, the line of your Contract that ordered it. A step no field
> ordered is a decision you have not defended yet."*
> — a **replication editor** who reruns other researchers' code for a living

Three terms carry this section. Meet them before any code runs.

- **Pipeline**: the chain of code steps that carries raw data to a reported
  result. Example: load the turnout file, keep the outcome and treatment
  columns, difference the two arm means.
- **Declared analysis**: the analysis your Research Contract committed you to
  before you saw any results, written in words. Example: "the difference in 2014
  turnout between the contacted and control arms, over all voters in the file."
- **Contract-to-code map**: the explicit pairing of each Contract field with the
  pipeline step that implements it. Example: the frame field pairs with the
  transform step, and the map asks that step one question: who ordered every
  dropped row?

The map exists because of a failure with a name you know from the AI weeks. A
**silent scope change** is an edit that quietly changes which cases a number is
computed over, so the code answers a different question than the one you asked.
The colleague's step 2 is exactly that: one innocent-looking filter, and an
8,375-voter declaration became a 379-voter answer with the opposite sign.

> **A question that often comes up here:** *"My code already runs. Why draw a
> map of it?"* Because running and matching are different tests. Execution
> checks that the code is legal Python; the map checks that it is *your
> analysis*. A green cell proves the code computed something. Only the map
> tells you whether that something is the quantity you declared, over the cases
> you declared. Every audit this week starts from the map, not from the
> number.

The next cell draws the map as a picture. The pipeline runs left to right along
the bottom; the Research Contract sits above it, ordering every step. The gray
node is the intruder to watch for: the edit that no Contract field ordered.

**What to expect when you run it:** a diagram with the three pipeline steps
flowing into a result and then a claim, three arrows coming down from the
Contract, and one warning arrow sneaking into the transform step.

In [ ]:
# The Contract-to-code map: the declaration orders every step of the pipeline.
# (networkx is imported once in the setup cell.)
G = nx.DiGraph()
pos = {
    "Research\nContract\n(declared analysis)": (2.0, 2.3),
    "LOAD\n(raw data)":            (0.0, 1.0),
    "TRANSFORM\n(analysis table)": (2.0, 1.0),
    "ESTIMATE\n(the quantity)":    (4.0, 1.0),
    "RESULT\n+ uncertainty":       (6.0, 1.0),
    "CLAIM\n(the sentence)":       (8.0, 1.0),
    "Silent scope change\n(the edit no field ordered)": (0.0, 2.3),
}
edges = [
    ("LOAD\n(raw data)", "TRANSFORM\n(analysis table)"),
    ("TRANSFORM\n(analysis table)", "ESTIMATE\n(the quantity)"),
    ("ESTIMATE\n(the quantity)", "RESULT\n+ uncertainty"),
    ("RESULT\n+ uncertainty", "CLAIM\n(the sentence)"),
    ("Research\nContract\n(declared analysis)", "LOAD\n(raw data)"),
    ("Research\nContract\n(declared analysis)", "TRANSFORM\n(analysis table)"),
    ("Research\nContract\n(declared analysis)", "ESTIMATE\n(the quantity)"),
    ("Silent scope change\n(the edit no field ordered)", "TRANSFORM\n(analysis table)"),
]
G.add_edges_from(edges)

fig, ax = plt.subplots(figsize=(11, 5))
nodes = list(pos.keys())
fills = ["#FCE4B8", "#DCE6F1", "#DCE6F1", "#DCE6F1", "#DCE6F1", "#DCE6F1", "#EAEAEA"]
lines = ["#B8860B", "#4C72B0", "#4C72B0", "#4C72B0", "#4C72B0", "#4C72B0", "#888888"]
for n, fc, ec in zip(nodes, fills, lines):
    nx.draw_networkx_nodes(G, pos, nodelist=[n], node_color=fc, edgecolors=ec,
                           linewidths=1.8, node_size=6000, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=8, ax=ax)
nx.draw_networkx_edges(G, pos, arrowstyle="-|>", arrowsize=16, edge_color="#555555",
                       width=1.5, node_size=6000, ax=ax)
ax.set_title("The Contract orders every pipeline step; the map is how you catch the edit it never ordered")
ax.set_xlim(-1.4, 9.4)
ax.set_ylim(0.2, 3.2)
ax.axis("off")
plt.tight_layout()
plt.show()

print("✓ Contract-to-code map drawn.")
print("  Bottom chain: LOAD -> TRANSFORM -> ESTIMATE -> RESULT -> CLAIM.")
print("  The Contract points into the three steps it orders; reconciliation walks those arrows.")
print("  The gray arrow is the colleague's step 2: an edit no Contract field ordered.")

**Reading the diagram.** Follow the bottom chain first: data becomes an analysis
table, the table becomes one number, the number becomes a sentence. Now follow
the arrows from the Contract. Each of the three code steps answers to a field
of the declaration, and reconciling the pipeline means walking those arrows one
at a time: is the loaded file the documented one, did the transform keep the
declared cases, is the estimate the declared quantity? The gray arrow has no
Contract field behind it. That is what makes it a silent scope change instead
of a design decision.

*In plain terms, the picture says your Contract is the boss of every line of
code, and any line without a boss is where your answer quietly changes.*

**The Contract-to-pipeline worksheet.** This is the map as a table. You will
fill the middle column for the demonstration today, and for your own project at
Friday's studio.

| Contract field | Pipeline step that implements it | The question you ask of the code |
|---|---|---|
| Data source (your M5 governance record) | LOAD | Is this the documented file, at the documented shape? |
| Frame (which cases) | TRANSFORM | How many rows in, how many out, and who ordered every drop? |
| Outcome measure | TRANSFORM | Is the declared column the one the code selects? |
| Quantity of interest | ESTIMATE | Is this arithmetic the declared quantity, not a lookalike? |
| Uncertainty form | ESTIMATE (Lecture 2) | Which uncertainty statement does the route provide? |
| The claim | after the pipeline | Does the sentence stay inside what the output licenses? |

## 2. Build the Pipeline, Step by Step

**Guiding question:** *can you predict what a declared pipeline will print before
you run it, and reconcile what it printed after?*

You will now build the demonstration pipeline on real data. The `foos_etal` file
is a **real** UK get-out-the-vote field experiment: chance assigned some voters
to a door-to-door contact arm and left others as controls, and whether each
voter turned out in 2014 was recorded afterward. Next week and Week 9 ask what
such an experiment licenses you to claim; this week it is simply a governed,
documented dataset on which to practice execution.

Here is the demonstration **Research Contract card**, declared before any code
ran. Every step you are about to run answers to one of its fields.

| Field | Declaration |
|---|---|
| Question | Did 2014 turnout differ between the contacted and control arms of this UK experiment? |
| Quantity of interest | difference in mean turnout, contacted minus control, in percentage points |
| Frame | all 8,375 voters in `foos_etal.csv`, the governed course file |
| Outcome measure | `marked_register_2014` (1 = the marked register shows a vote) |
| Comparison | contacted arm (`treat = 1`) vs control arm (`treat = 0`) |
| Uncertainty form | two-proportion standard error and a 95% interval (attached in Lecture 2) |

The pipeline below is a **route-neutral pipeline skeleton**: three small steps,
each one owned by a Contract field. Your own project keeps the same skeleton and
swaps in your route's own estimate step, whether that is a survey summary, a
model's held-out error, or an arm comparison like this one.

> **A question that often comes up here:** *"The tool that wrote my code says it
> does what I asked. Isn't that enough?"* No, and the reason is structural: the
> tool reviews its own work with the same fluency it wrote it, and it answers
> from the code text, not from your Contract. The reconciliation is yours. The
> working habit this week installs: paste the declaration next to the code and
> check the pairing yourself, step by step, against the printed receipts.

### 🔮 Predict First

You are about to run the three-step pipeline. Before you run it, commit three
predictions in writing. First: after the transform step, how many of the 8,375
rows will survive into the analysis table? (The declared frame says every
voter; predict whether the code will honor that.) Second: will the contacted
arm's turnout land above or below the control arm's? Third: name a rough size
for the gap, in percentage points. Do not peek at the output first. You are
predicting a reveal.

### YOUR ANSWER HERE:

**Rows surviving the transform (out of 8,375):**

**Direction of the gap (contacted above / below control):**

**Rough size in percentage points:**

---

### 🛠️ Run the Study: build the declared pipeline

Run the cell below. It executes the three steps and prints a receipt for each
one: the loaded shape, the rows in and out of the transform, and the arm sizes,
arm means, and estimate. Read each receipt against the Contract card before you
read anything else.

**🔴 Live in class: we run this one together.**
**Before you ask:** write one sentence naming the step where you think a silent
scope change would be easiest to miss in this pipeline.

> 💡 **AI Prompt:** "Here is a declared analysis and the pipeline that claims to
> implement it. Declaration: 'the difference in 2014 turnout between the
> contacted arm and the control arm, over all 8,375 voters in the experiment
> file, in percentage points.' Code: [paste the next cell]. For each of the
> three steps, name the Contract field it implements. Then tell me the first
> place in this code where the estimand could silently change if a line were
> edited, and give me a one-line check that would catch that change. Do not
> evaluate style; evaluate correspondence."
>
> **After running, verify (counters *silent scope change*: a step that quietly
> narrows which cases the number is computed over answers a different question
> than the declared one):**
> - [ ] Check the mapping against your printout: the transform receipt must read
>   rows in 8,375, rows out 8,375, dropped 0. If the tool praises a step for
>   "cleaning the data," ask which cases it would remove and which Contract
>   field ordered the removal.
> - [ ] Confirm the numbers it quotes match what your cell printed (arms of
>   6,104 and 2,271, a gap near +3.4 points), not values it guessed.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# THE DEMONSTRATION PIPELINE — three steps, each annotated with the Contract field it implements.

OUTCOME = "marked_register_2014"    # Contract: outcome measure (1 = the register shows a vote)
GROUP   = "treat"                   # Contract: comparison (contacted = 1, control = 0)

def step1_load():
    # LOAD — Contract fields: data source + frame (the governed file, every voter)
    df = load_course_data("foos_etal.csv")
    assert df.shape == (8375, 5), "unexpected shape — stop and investigate before anything else"
    return df

def step2_transform(df):
    # TRANSFORM — Contract fields: outcome measure + frame (keep every case; no silent drops)
    return df[[GROUP, OUTCOME]].dropna()

def step3_estimate(table):
    # ESTIMATE — Contract field: quantity of interest (difference in mean turnout between arms)
    arm_means = table.groupby(GROUP)[OUTCOME].mean()
    return arm_means[1] - arm_means[0]

foos = step1_load()
print(f"✓ STEP 1 LOAD      : {foos.shape[0]} rows, {foos.shape[1]} columns from foos_etal.csv")

pipe_table = step2_transform(foos)
print(f"✓ STEP 2 TRANSFORM : rows in {len(foos)}, rows out {len(pipe_table)}, dropped {len(foos) - len(pipe_table)}")

arm_sizes = pipe_table[GROUP].value_counts()
arm_means = pipe_table.groupby(GROUP)[OUTCOME].mean()
first_estimate = step3_estimate(pipe_table)
print(f"✓ STEP 3 ESTIMATE  : contacted arm {arm_sizes[1]} voters, turnout {arm_means[1]:.4f}")
print(f"                     control arm  {arm_sizes[0]} voters, turnout {arm_means[0]:.4f}")
print(f"\n  the declared quantity: contacted minus control = {first_estimate*100:+.2f} percentage points")
print("  (a point estimate only — its uncertainty statement is attached in Lecture 2)")

In [ ]:
# Self-check: the pipeline honors the declaration, and nothing was silently dropped.
assert len(pipe_table) == 8375, "the declared frame is every voter in the file"
assert arm_sizes[1] == 6104 and arm_sizes[0] == 2271, "arm sizes changed — investigate"
assert 0.030 < first_estimate < 0.039, "the estimate drifted from the expected +3.4 points"
print("✓ Self-check passed: every voter kept (8,375), arms of 6,104 and 2,271,")
print(f"  and the declared quantity computes to {first_estimate*100:+.2f} points.")
print("  Executed is not yet correct: Lecture 2's tests decide what this number has earned.")

### 🔍 Read the Evidence: reconcile the code against the declaration

The receipts are on screen. In the cell below, walk the Contract-to-code map for
this run. First: for each of the three steps, one line stating the Contract
field it implements and the receipt that proves it (the shape check, the
rows-in-rows-out count, the arm means). Second: state what the +3.41 does and
does not establish so far. It is the declared quantity, computed over the
declared frame; it carries no uncertainty statement yet, and no verification of
the code beyond a clean run. Third: return to the colleague's pipeline from the
puzzle and write the one-sentence audit verdict you would send back, naming the
step and the fix.

### YOUR ANSWER HERE:

**Step 1 (Contract field + the receipt that proves it):**

**Step 2 (Contract field + the receipt that proves it):**

**Step 3 (Contract field + the receipt that proves it):**

**What +3.41 does and does not establish so far:**

**My audit verdict on the colleague's pipeline:**

---

### 📝 Practice: find the first estimand change

*(In class: answers aloud, a couple of minutes; writing them down is optional.)*

A colleague declares: "the share of all 12,000 respondents in the survey file
who rate trust in local government at 5 or higher, on the 1-to-7 scale." Their
pipeline runs green in four steps. Name the **first** step where the estimand
changes, and what the printed number is actually a number of.

1. `df = load("survey.csv")` (12,000 rows arrive)
2. `df = df.dropna(subset=["trust"])` (11,140 rows remain)
3. `df = df[df["age"] >= 30]` (7,900 rows remain)
4. `share = (df["trust"] >= 5).mean()` (prints `0.61`)

### YOUR ANSWER HERE:

**First step where the estimand changes:**

**What the printed 0.61 is actually a number of:**

---

### ⚖️ Make a Design Choice: when is a code fix a Contract version?

*(In class: commit to one option in a single written line and be ready to defend
it aloud; the full write-up is optional depth.)*

Mid-build, you find a mistake in your own pipeline. Which policy do you adopt
for your project?

- **A.** Code is code: fix anything silently, since only the final version ships.
- **B.** Fix freely, but any edit that changes the computed quantity or the case
  set gets a Contract version note, and every fix gets a ledger row.
- **C.** Freeze the pipeline: never edit code after the first run, so nothing
  can drift.

**My choice:**

**Reason:**

**Risk:**

### 🎯 Take It to Your Project

One sentence, in class: name the first line of your own Contract-to-code map,
the Contract field paired with the pipeline step where your estimand is most at
risk. Write it below, then add the same line to your Research Project Dossier.

**The spine you build at Friday's studio (M6):** the full pipeline map from the
worksheet · the runnable three-step scaffold on your governed data · the
colleague-audit habit turned on your own code. The M6 brief collects all of it;
your AI assistant helps there, and the reasoning stays yours.

**My line:**

### 🛡️ Defend Your Decision

Close the lecture out loud. In the three lines below, state the claim you now
hold from today, the evidence that justifies it, and the uncertainty it still
carries.

**Claim:**

**Justification:**

**Uncertainty:**

### 📒 AI Research Ledger

Before you leave the room, add today's row to your AI Research Ledger: task
delegated · tool used · prompt · output summary · decision · verification
method · remaining concern · responsible researcher. One honest row is enough,
and if you delegated nothing to AI today, log that decision too.

You have built and reconciled a pipeline: every step now answers to a Contract
field, and a point estimate of +3.41 sits on screen. Lecture 2 attaches what
that number still lacks before anyone may call it a result: its uncertainty
statement, a real verification of the delegated code, and the records that let
a stranger check the claim.

---

---

### ⏸ Optional depth from here

**Today's lecture path is complete.** Anything between this line and the next lecture heading is optional depth: run it if you want to push the ideas further. Nothing below this line is required, and any 🏠-marked prompt above it is optional too.

## 3. Optional Depth: What One Changed Line Does to the Answer

**Guiding question:** *if the skeleton stays identical and one line changes, how
far can the answer move?*

The experiment behind `foos_etal` assigned treatment with **unequal
probabilities** across wards, and the file ships a `weights` column that
corrects for it. A **design weight** is a number that restores each case's
intended share of the analysis when the design gave some cases a higher chance
of landing in an arm. The published study analyzes the experiment weighted; our
classroom Contract declared the simple unweighted difference, and taught it as
the classroom-simple version.

Run the cell below. It swaps exactly one line, the estimate step, for a
weighted version, and prints both answers side by side.

**🏠 Optional depth.** Run this on your own if you want to go deeper.
**Before you ask:** write one sentence predicting the direction the weighted
number moves, and why a one-line change is enough to move it.

> 💡 **AI Prompt:** "A field experiment assigned treatment with unequal
> probabilities across areas and ships a design-weights column. My unweighted
> difference in means is +3.41 points; the weighted difference is +2.83 points.
> Explain what a design weight corrects, why the two estimates differ, and
> which of the two implements a Contract that declares 'the design-corrected
> difference in turnout between arms.' Then state what a Contract that wants
> the unweighted number would have to say instead."
>
> **After running, verify (counters *plausible-but-wrong-method*: an estimator
> that sounds standard can still be the wrong one for the declared quantity):**
> - [ ] Confirm the two numbers it discusses are yours (+3.41 and +2.83), read
>   from your printout, not re-derived approximations.
> - [ ] Check its bottom line against the worksheet: the estimator belongs in
>   the Contract's quantity field, so switching between weighted and unweighted
>   is a Contract version, never a silent swap.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# OPTIONAL DEPTH — the same skeleton with ONE changed line: a weighted estimate step.
def step3_estimate_weighted(df):
    # the design assigned arms with unequal probabilities; weights restore each case's intended share
    p1 = np.average(df.loc[df[GROUP] == 1, OUTCOME], weights=df.loc[df[GROUP] == 1, "weights"])
    p0 = np.average(df.loc[df[GROUP] == 0, OUTCOME], weights=df.loc[df[GROUP] == 0, "weights"])
    return p1 - p0

weighted_estimate = step3_estimate_weighted(foos)
print(f"  unweighted estimate step : {first_estimate*100:+.2f} points  (the classroom declaration)")
print(f"  weighted estimate step   : {weighted_estimate*100:+.2f} points  (the design-corrected estimator)")
print(f"  the one-line change moved the answer by {(weighted_estimate - first_estimate)*100:+.2f} points")
print()
print("  Same file, same frame, same comparison. The estimator lives in the Contract's")
print("  quantity field, so this switch is a declaration decision, never a silent code edit.")

**Reading the output.** The weighted step lands near +2.83 points, about 0.6
points below the unweighted +3.41. Nothing about the data changed; one line of
the estimate step did. That is the whole argument for the Contract-to-code map
in miniature: two defensible pipelines, two different answers, and the only
thing that decides between them is which one your declaration actually ordered.
When your own route offers estimator choices like this, the choice belongs in
the Contract, made before results are seen, with a version note if it ever
changes.

---

# Lecture 2

🗺️ **Today's frame (Wednesday, 7 / 23 / 12 / 8):** open with the 🧩 challenge, a spoken 📝 **Practice** retrieval drill, and 🔮 **Predict First** · attack the problem with 🛠️ **Run the Study**, 🔁 modifying the prompt and 🔬 interrogating what comes back · verify with 🔍 **Read the Evidence** · defend ⚖️ **Make a Design Choice** to your peers, AI closed at the 🧑‍⚖️ checkpoint · close with 🎯 **Take It to Your Project**, 🛡️ **Defend Your Decision**, and your 📒 ledger row. All seven moves happen in class; 🏠-marked items and everything below the ⏸ line are optional depth.

### 🎤 SRL Lead Brief

*This lecture opens with its Student Research Lead. If this is your slot, this
brief is your backbone. If not, read along: this is how the room will run.*

**Your mission.** By minute 50 everyone can state what must be attached to a
point estimate before it counts as a result, and can verify one piece of
delegated code with a known-answer test instead of an explanation.

**Run of show (Wednesday: 7 / 23 / 12 / 8).**

| Min | Section | What you do |
|---|---|---|
| 0–7 | Retrieval challenge | Present the colleague result card below, cold. Everyone commits in writing: what is this card missing before its sentence can be defended? Then run the spoken retrieval drill. |
| 7–30 | Applied AI laboratory | Run sections 4 and 5 with the room: attach the uncertainty, then put the live verification prompt to your AI and hold its explanation against the printout. Close the lab on the known-answer test and its lookalike. |
| 30–38 | Peer defense | Pair people off: each defends their one-sentence result, while the partner attacks the missing pieces, no uncertainty, no trace, no verified line. |
| 38–42 | Synthesis | Pull the room's findings into the rule of the day, then hand your instructor the accuracy lock: every result sentence carries estimate, uncertainty, frame, and boundary. |
| 42–50 | Project transfer | Everyone records the decision and a ledger entry; close on a spoken Claim Ticket. |

**Three questions that keep the room thinking.**

1. *"Is +3.41 points a result?"* Listen for: not yet. It becomes one when its
   uncertainty statement is attached and the number traces to an output a
   stranger can check.
2. *"The tool explained every line beautifully. What did that verify?"* Listen
   for: nothing about the numbers. An explanation is a story about the code;
   the known-answer test is evidence about it.
3. *"What does the interval from +1.0 to +5.8 let you say that +3.41 alone does
   not?"* Listen for: the span of gaps compatible with the data, so the claim
   survives even if the point lands elsewhere inside it.

**One AI trap to watch for.** Someone will paste their pipeline and ask "is my
result correct?" The reply is a confident verdict produced without touching a
single number: it praises the method, blesses the estimate, and invents no
check. When it appears, ask the machine which printed value it verified and
how. The honest answer is none, and that is the moment the known-answer test
earns its place.

**Checkpoints.** By minute 7, everyone has written what the colleague card is
missing. By minute 30, the interval is on screen, the known-answer test has
passed, and the lookalike has been caught. By minute 42, the accuracy lock has
run: every result sentence in the room carries its four parts.

**Make it yours.** The frame, the challenge, and the checkpoints are fixed; the
staging is yours. Restage the colleague card as a poster you tape to the wall
and let the room attack it; or run the peer defense as a journal desk, where
each pair decides accept, revise, or reject on the partner's sentence; or open
with a result from your own field that shipped without uncertainty and what it
cost. In your prep script, name one thing you are adding that this brief does
not contain.

**Prep.** Start about a week ahead. Submit your preparation script or notebook
two days ahead. The full guide is the Student Research Lead handbook in the
course materials.

### 🧩 Research Puzzle

*(Your research lead opens the lecture with this. Think it through and commit an
answer before we go further. No AI yet.)*

Your instructor issues a second fragment from the colleague's dossier.

> **SIMULATED CASE · TRAINING MATERIAL · NOT EVIDENCE**
>
> *Milestone excerpt:* "Door-to-door contact raised turnout by 3.4 percentage
> points (n = 8,375). The pipeline ran top to bottom with no errors, and the
> code was written and double-checked by my AI assistant. Result final."

The sentence is polished, the number is plausible, and nothing crashed. Here is
the question on the table: **what is this card missing before its sentence can
be defended in public?** Commit a written list before we go further. Two
attachments and one label are absent, and one phrase in the excerpt is doing a
job it cannot do. Name as many as you can. Turning a polished number into a
defensible result is the whole job of Lecture 2.

### 📝 Practice: retrieval drill

Before any AI opens today: from memory, in about a minute, state (1) the three
pipeline steps and the Contract field that owns each, (2) one claim Monday's
work permits, and (3) one claim it does not permit. Answers run aloud; nothing
written is required.

## 4. Execute and Attach the Uncertainty

**Guiding question:** *what first result does the declared analysis produce, and
where is its uncertainty?*

> *"You are showing me one number and asking me to believe it. Show me instead
> the range of numbers your data cannot tell apart. If your claim only lives at
> the exact point you printed, you do not have a claim."*
> — a **journal reviewer**, returning a manuscript that reported a bare point estimate

Three terms carry the rest of this week.

- **Point estimate**: the single best number your pipeline produces for the
  declared quantity. Example: +3.41 percentage points.
- **Uncertainty statement**: the honest range around that number, in the form
  your Contract declared. Example: a **standard error** (the typical distance
  between an estimate and the truth it aims at) of 1.23 points, giving a 95%
  interval from +1.0 to +5.8.
- **Provisional result**: a result you report while its verification is still
  pending, labeled as such. Example: this week's number, which next week's
  clean restart (M7) either confirms or turns into a finding about the
  pipeline.

The studio rule this section installs: **a result reported without uncertainty
is not yet a result.** Routes differ in the form the uncertainty takes, an
interval here, a spread of held-out errors on a prediction route, a resampling
band on a survey summary. No route is excused from attaching it.

> **A question that often comes up here:** *"My project is a survey route, not
> an experiment. Does this lecture apply to me?"* Fully. The pipeline skeleton,
> the uncertainty requirement, the verification tests, and the records are
> route-neutral; only the arithmetic inside the estimate step and the form of
> the uncertainty statement change. Your route lesson names the form; this
> lecture is where you practice attaching it.

### 🔮 Predict First

You are about to attach the two-proportion standard error and 95% interval to
Monday's +3.41. Before running, commit two predictions in writing. First: will
the interval exclude zero, or reach across it? Second: name a rough width for
the interval, in points from end to end. One sentence of reasoning. Do not peek
at the output first.

### YOUR ANSWER HERE:

**Excludes zero or reaches across it:**

**Rough width, end to end (in points):**

**One sentence of reasoning:**

---

### 🛠️ Run It Live: attach the uncertainty

Run the cell below. It computes the standard error and 95% interval for the
declared difference, assembles the **result-with-uncertainty object** your M6
package will carry, and plots the estimate against the no-difference line. Read
the interval before you read the next markdown cell.

**🔴 Live in class: we run this one together.**
**Before you ask:** the code below was the kind of thing you delegate. Write one
sentence naming the check you would demand before trusting its numbers.

> 💡 **AI Prompt:** "You are reviewing analysis code you did not write. Here it
> is, with what it printed: [paste the next cell and its output]. Explain, line
> by line, what each quantity is. Then give me one independent way to confirm
> the standard error and the interval without rerunning this exact code, and
> name the single line whose failure would most change the reported result."
>
> **After running, verify (counters *confident fabrication*: a fluent
> line-by-line walkthrough can describe numbers the code never printed):**
> - [ ] Confirm every number the explanation quotes matches your printout
>   exactly: the +3.41, the 1.23, and the interval endpoints near +1.0 and
>   +5.8. One invented decimal disqualifies the walkthrough.
> - [ ] Run its independent confirmation yourself, by hand or in a fresh cell.
>   If it lands away from your printout, trust the printout and investigate
>   before accepting either.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# ATTACH THE UNCERTAINTY — the point estimate becomes a result-with-uncertainty object.
t1 = pipe_table.loc[pipe_table[GROUP] == 1, OUTCOME]
t0 = pipe_table.loc[pipe_table[GROUP] == 0, OUTCOME]
diff = t1.mean() - t0.mean()
se = np.sqrt(t1.mean() * (1 - t1.mean()) / len(t1)
             + t0.mean() * (1 - t0.mean()) / len(t0))
ci_lo, ci_hi = diff - 1.96 * se, diff + 1.96 * se

result_v1 = {
    "quantity": "difference in 2014 turnout, contacted minus control (percentage points)",
    "frame":    f"all {len(pipe_table)} voters in foos_etal.csv",
    "estimate_pts": round(float(diff) * 100, 2),
    "se_pts":       round(float(se) * 100, 2),
    "ci_pts":       (round(float(ci_lo) * 100, 1), round(float(ci_hi) * 100, 1)),
    "status":   "provisional (pending the clean-restart verification, M7)",
}
print("RESULT v1 — the result-with-uncertainty object:")
for k, v in result_v1.items():
    print(f"  {k:>12} : {v}")
print(f"\n  one line: contacted minus control = {diff*100:+.2f} points, 95% interval "
      f"[{ci_lo*100:+.1f}, {ci_hi*100:+.1f}] points (excludes 0: {ci_lo > 0})")

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.errorbar([0], [diff * 100], yerr=[[1.96 * se * 100]], fmt="o", color="#4C72B0",
            capsize=8, markersize=10, linewidth=2, label="estimate with 95% interval")
ax.axhline(0, color="#C44E52", linestyle="--", linewidth=1.5, label="no difference (0)")
ax.set_xlim(-0.6, 0.6)
ax.set_xticks([0])
ax.set_xticklabels(["first result, version 1"])
ax.set_ylabel("Turnout difference, contacted minus control (points)")
ax.set_title("The first result with its uncertainty attached: the interval clears zero")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Self-check: the result object carries what M6 requires.
assert len(t1) == 6104 and len(t0) == 2271, "arm sizes changed — investigate"
assert 3.0 < diff * 100 < 3.9, "the point estimate drifted from the expected +3.4"
assert abs(se * 100 - 1.23) < 0.05, "the standard error drifted — recheck the formula"
assert ci_lo > 0, "the 95% interval should exclude zero here"
assert "provisional" in result_v1["status"], "the provisional label is part of the result"
print(f"✓ Self-check passed: {diff*100:+.2f} points, se {se*100:.2f}, "
      f"interval [{ci_lo*100:+.1f}, {ci_hi*100:+.1f}], labeled provisional.")
print("  A result reported without uncertainty is not yet a result; this one now qualifies,")
print("  pending verification of the code that produced it — next section.")

### 🔍 Read the Evidence: write the one-sentence result

The interval is on screen: the data cannot tell apart gaps from about +1.0 to
+5.8 points, and zero sits outside that span. In the cell below, write the
**one-sentence result** your M6 package would carry, containing all four parts:
the estimate, its uncertainty, the frame it describes, and the design boundary
(what this number does not cover). Then write the sentence you must refuse: the
version that reports the point alone, or upgrades it beyond the frame.

> **A question that often comes up here:** *"The interval excludes zero. Doesn't
> that settle it?"* It settles less than it seems. It says chance alone is a
> poor explanation for a gap this size in this file. It does not verify the
> code that produced it, does not survive a stale-kernel mistake, and does not
> reach beyond these 8,375 voters. That is why the result stays labeled
> provisional until the clean restart, and why the verification section comes
> next.

### YOUR ANSWER HERE:

**My one-sentence result (estimate + uncertainty + frame + boundary):**

**The sentence I refuse to write:**

---

## 5. Verify the Delegated Code, Then Trace the Result

**Guiding question:** *how do you verify code you did not write, and leave a
trail a stranger could follow?*

You delegated the writing; you never delegate the verdict. The M6 brief names
the two checks, and both live in your notebook next to the code they verify.

- **Known-answer test**: run the delegated code on a tiny input where you
  already know the right output, and confirm it returns exactly that. Example:
  feed your arm-difference step two three-row arms you can average in your
  head.
- **Line review**: read every AI-written line and say in your own words what it
  does. Any line you cannot explain gets rewritten or removed; you cannot
  defend a pipeline you cannot narrate.

A known-answer test is a **sensitive diagnostic**: it exists to detect a
pipeline that computes a different quantity than the declared one, and a pass
lowers your concern only as far as the test is sensitive. It clears exactly the
step it exercised, not the whole notebook. The standing rule from the AI weeks
applies with full force here: **code running without errors is not the same as
code being correct.** Verify the number, not the green check.

> **A question that often comes up here:** *"The explanation of every line was
> right. Isn't the code verified?"* No. An explanation is a story about the
> code, and stories can be accurate about structure while wrong about output.
> The known-answer test is evidence: it makes the code commit to a number you
> can check against arithmetic you did yourself. The two differ exactly when it
> matters, as the lookalike below shows.

**What to expect when you run the next cell:** a toy world where the hand answer
is +33.33 points, a pass from the real estimate step, and a lookalike function
that also runs green while computing a different quantity.

In [ ]:
# THE KNOWN-ANSWER TEST — feed the pipeline a world where you already know the answer.
toy = pd.DataFrame({
    GROUP:   [1, 1, 1, 0, 0, 0],
    OUTCOME: [1, 1, 0, 1, 0, 0],   # contacted arm: 2 of 3 voted; control arm: 1 of 3
})
hand_answer = (2/3 - 1/3) * 100     # computable in your head: +33.33 points

test_result = step3_estimate(toy) * 100     # the SAME estimate step the real run used
print(f"  hand answer            : {hand_answer:+.2f} points")
print(f"  estimate step on toy   : {test_result:+.2f} points")
assert abs(test_result - hand_answer) < 1e-9, "the estimate step does not reproduce the hand answer!"
print("✓ Known-answer test PASSED: the estimate step computes the declared quantity.")
print()

# A LOOKALIKE that also runs green: the share of VOTERS who were contacted.
def lookalike_estimate(table):
    return table.loc[table[OUTCOME] == 1, GROUP].mean()

wrong_toy  = lookalike_estimate(toy) * 100
wrong_real = lookalike_estimate(pipe_table) * 100
print(f"  lookalike on toy       : {wrong_toy:+.2f} points  (hand answer says {hand_answer:+.2f} -> CAUGHT)")
print(f"  lookalike on real data : {wrong_real:.1f}   (plausible-looking, and a different quantity)")
print("⚠️ Both functions execute without a single error. Only the known-answer test tells them apart.")

**Reading the output.** The real estimate step returned +33.33 on the toy world,
matching the arithmetic you can do in your head, so that step is verified for
the quantity it claims to compute. The lookalike returned +66.67 on the same
toy: it runs, it produces a tidy number on the real data too (74.2, the share
of 2014 voters who were in the contacted arm), and it is not the declared
quantity. Nothing in the console distinguishes the two. The hand answer does.
That is the whole case for building the toy before you trust the pipeline, and
it is the first of the two checks your M6 verification record reports. The
second, the line review, you practice now.

### 🔁 Modify the Prompt

*(In class: one modification, one prediction, one rerun; deeper variations are
optional depth.)*

The live prompt above verified the demonstration's uncertainty code. Now aim
the same verification at your own project. First, write your declared quantity
and frame yourself, in one sentence each. Do not let AI draft them. Then adapt
the review prompt so it interrogates *your* pipeline, naming your columns and
your steps.

> **Base prompt (the review, from above):** "You are reviewing analysis code you
> did not write. Here it is, with what it printed: [code + output]. Explain,
> line by line, what each quantity is. Then give me one independent way to
> confirm the key numbers without rerunning this exact code, and name the
> single line whose failure would most change the reported result."

Rewrite it so it names *your* declared quantity, *your* frame, and the step you
flagged on Monday as most at risk. In the cell below: paste your declaration,
paste your adapted prompt, predict the single riskiest line it will name, then
run it and note whether your prediction held.

### YOUR ANSWER HERE:

**My declared quantity and frame (AI did not draft this):**

**My adapted review prompt:**

**The riskiest line I predict it will name:**

**What it actually named, and what I did about it:**

---

### 🔬 Interrogate the Output

*(In class: raise the sharpest challenge and check it; the full written
interrogation is optional depth.)*

Your AI just reviewed delegated code, line by line, and proposed an independent
confirmation. Do not accept the review on fluency. Interrogate it against the
printouts, using four checks, and answer each in the cell below.

- **Claims:** does every number the review quotes match your output exactly
  (+3.41, se 1.23, interval +1.0 to +5.8)? A review that misquotes even one
  printed value has been describing code it imagined.
- **Assumptions:** what does its independent confirmation assume, and does that
  assumption hold for your data? The two-proportion formula, for instance,
  assumes a binary outcome and independent observations; name what yours needs.
- **Missing information:** what did the review not mention? A strong review of
  this pipeline flags the `weights` column sitting unused in the file. A review
  that missed it missed the most consequential line *not* in the code.
- **Overstatements:** find the sentence that sounds more settled than the
  evidence, for example any suggestion that a clean explanation plus a clean
  run makes the result final. The provisional label survives every review
  until the clean restart passes.

### YOUR ANSWER HERE:

**Claims (every quoted number checked against my printout):**

**Assumptions (what the confirmation needs, and whether my data provides it):**

**Missing information (what the review never mentioned):**

**Overstatements (the exact words):**

---

With the code verified, two small records make the result checkable by someone
who was not in the room. Both are part of your M6 package, and both take
minutes now versus hours to reconstruct later.

- **Environment record**: the snapshot of what your pipeline ran on: language
  and library versions, the seed, the dataset and its shape, and a fingerprint
  of the data itself. Example: the printout below, which is version 1 of the
  record your project carries forward.
- **Claim-output trace**: one row per claim you plan to make, linking the
  sentence to the exact output that supports it, the verification that checked
  it, and the environment it ran in. Example: the single row below, for the
  one-sentence result you wrote in section 4.

**What to expect when you run the next cell:** the v1 environment record printed
field by field, then the claim-output row for today's result.

In [ ]:
# THE V1 RECORDS — the environment record and the claim-output trace row.
import sys, platform
import matplotlib

data_fingerprint = int(pd.util.hash_pandas_object(foos).sum()) % 10**12

env_record = {
    "python":           platform.python_version(),
    "numpy":            np.__version__,
    "pandas":           pd.__version__,
    "matplotlib":       matplotlib.__version__,
    "seed":             SEED,
    "dataset":          "foos_etal.csv",
    "rows x cols":      f"{foos.shape[0]} x {foos.shape[1]}",
    "data fingerprint": data_fingerprint,
}
print("ENVIRONMENT RECORD, version 1 (what a stranger needs to rerun this):")
for k, v in env_record.items():
    print(f"  {k:>16} : {v}")

claim_output_row = {
    "claim":       "contacted-arm turnout 3.41 points above control (95% interval +1.0 to +5.8), provisional",
    "traces to":   f"the uncertainty cell: diff {result_v1['estimate_pts']:+.2f}, interval {result_v1['ci_pts']}",
    "verified by": "known-answer test on the estimate step + line review of the delegated code",
    "environment": f"python {platform.python_version()}, pandas {pd.__version__}, seed {SEED}",
}
print("\nCLAIM-OUTPUT TRACE (one row per claim you plan to make):")
for k, v in claim_output_row.items():
    print(f"  {k:>16} : {v}")

print("\n✓ Records built. Copy both into your M6 pipeline notebook, next to the result they cover.")

### 🧑‍⚖️ Human-Only Checkpoint

*(In class: AI closed, one decision, one line of reasoning.)*

Close your AI for this one. No AI, no notebook search. This is a never-delegate
decision: what your project's public description may claim is yours to set and
defend. This week your **URC abstract** (the short public description of your
project for the Undergraduate Research Conference) clears an internal gate at
Friday's studio, and the gate has one non-negotiable rule: **the abstract may
only use claims the current result licenses.** Not the result you expect by
the conference. The one you have today, with its provisional label doing the
honesty work. In the cell below, write, in your own words:

1. Your project's current first-result state, one line, honestly labeled
   (a provisional number with its uncertainty, or "no executed result yet,"
   which is a legitimate state to describe).
2. **Two sentences, the pair the M6 gate checks.** First, the abstract sentence
   your current evidence licenses, worded so no verification outcome can make
   it retroactively false. Second, the named overreach you refuse: the tempting
   sentence that promises a finding your route has not yet earned, written out
   so you can spot it in your own draft.

### YOUR ANSWER HERE:

**My current first-result state (honestly labeled):**

**Sentence 1, what my abstract may say today:**

**Sentence 2, the overreach I refuse:**

---

### ⚖️ Make a Design Choice: which sentence enters the abstract?

*(In class: commit to one option in a single written line and be ready to defend
it aloud; the full write-up is optional depth.)*

Your URC abstract draft needs its results sentence now, while your result is
provisional. Choose your policy:

- **A.** Write the sentence today's result licenses, provisional label included,
  and revise upward only if verification earns it.
- **B.** Write the sentence you expect to be true by the conference, so the
  abstract will not need edits later.
- **C.** Leave results out of the abstract entirely until M7 verifies the
  number.

**My choice:**

**Reason:**

**Risk:**

### 🎯 Take It to Your Project

One sentence, in class: name the piece of your M6 package you will build first
at Friday's studio. Write it below, then add the same line to your Research
Project Dossier.

**The spine you assemble at Friday's studio (M6, First executable analysis +
URC abstract internal gate):** the pipeline notebook that runs top to bottom on
your governed data · the first result with its uncertainty statement, labeled
provisional · the known-answer test and line review, recorded next to the code
they verify · the v1 environment record and claim-output row · the URC abstract
held inside what the current result licenses. The M6 brief collects all of it;
your AI assistant helps there, and the decisions stay yours.

**My line:**

### YOUR ANSWER HERE:

**My line (the M6 piece I build first, and why it is first):**

---

### 🛡️ Defend Your Decision

Defense #07 — the short ritual close, one line each:

1. **The claim I can defend:** one bounded sentence you would put your name on.
2. **Its boundary:** what today's evidence does not establish (name the frame
   your result describes and the verification it still awaits).
3. **My uncertainty and limitations:** the interval or spread you attached, and
   the provisional label, one line.
4. **AI check:** what you delegated, and the specific test that decided what
   you kept.

### YOUR ANSWER HERE:

**1. The claim I can defend:**

**2. Its boundary:**

**3. My uncertainty and limitations:**

**4. AI check (what I delegated, how I verified):**

---

### 📒 AI Research Ledger

Log every AI use from this notebook in the ledger. One worked row is filled in
as a model, and it models the discipline this week teaches: **the code was
delegated; the verdict was not.** This notebook offers four prompts, one live
per lecture and two optional-depth, so your ledger carries a row for each one
you actually ran, not a fixed count. The ledger is a graded habit, not
paperwork: it is how you show your work was verified.

| Task delegated | Tool used | Prompt | Output summary | Decision | Verification method | Remaining concern | Responsible researcher |
|---|---|---|---|---|---|---|---|
| Line-by-line review of the delegated uncertainty code | your AI | "You are reviewing analysis code you did not write: [code + output]. Explain each quantity and give one independent confirmation." | Correct walkthrough of the standard error and interval; proposed recomputing the two arm rates by hand; did not mention the unused weights column | Accepted the walkthrough after checking every quoted number; added the weights caveat myself | Known-answer test on the estimate step (+33.33 hand answer reproduced); hand recomputation of both arm means | The review verified this pipeline, not the clean-restart behavior M7 tests | *(your name)* |
|  |  |  |  |  |  |  |  |
|  |  |  |  |  |  |  |  |

---

## 6. Wrap-Up

Across two lectures you took your project across its widest gap so far: from an
analysis written in words to a running, checkable pipeline. You drew the
Contract-to-code map and used it to catch a colleague's silent scope change,
one filter that turned an 8,375-voter declaration into a 379-voter answer with
the opposite sign. You built the three-step skeleton, predicted its output,
and read every receipt against the declaration. Then you made the number a
result: you attached its **uncertainty statement**, labeled it **provisional**,
verified the delegated code with a **known-answer test** that caught a green-
running lookalike, reviewed it line by line, and left behind the environment
record and claim-output row that let a stranger check the claim.

> **"A clean run proves the code executed. Only the map, the uncertainty
> statement, and the trace prove what it executed. Attach the uncertainty,
> label the result provisional, and let your abstract say only what today's
> result licenses."**

Next week is the reckoning this week set up: your provisional number faces a
**clean restart** in a fresh runtime, where either every number reproduces or
the discrepancy becomes the finding (M7). Your M6, the first executable
analysis with its URC abstract internal gate, is worked on and submitted at
this week's Friday studio. Bring your Contract, your governed data, and the
scaffold you started today. This notebook companions **EDR|AI ch. 21, 'AI as
Programmer.'**

---

## 7. Sources & Provenance

**Provenance of this notebook:**
- *EDR|AI ch. 21 'AI as Programmer' | the quantity-and-frame discipline, the AI coding loop, a clean run is not a correct result, delegated-code verification | adapted (course-lab version of the chapter's discipline)*
- *foos_etal.csv | rdss package data | Foos et al. UK get-out-the-vote field experiment replication; analyzed as an unweighted difference in means with a two-proportion standard error and 95% interval (the study's own analysis is weighted; the optional-depth section teaches that exact caveat) | adapted (classroom-simple version)*
- *course replication module | the data-and-code lineage diagram, simplified into the Contract-to-code pipeline map | adapted*
- *fresh | the Contract-to-pipeline worksheet, the route-neutral pipeline skeleton, the known-answer test with its lookalike counterexample, the v1 environment record, and the claim-output trace row (seed 464) | newly-constructed-from-source-concept*

**Dataset attribution:** Dataset from the `rdss` package (Blair, Coppock &
Humphreys, MIT License), companion to *Research Design in the Social Sciences*
(2023).

**Readings:**
- Moreira, D. *Evidence-Driven Research in the Age of AI* (EDR|AI), ch. 21
  'AI as Programmer' (required):
  [the chapter](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/part4-credible-evidence/19-ai-as-programmer.html).
- Blair, G., Coppock, A., & Humphreys, M. (2023). *Research Design in the Social
  Sciences* (the dataset's companion book; recommended background). Free online:
  [book.declaredesign.org](https://book.declaredesign.org/).

---

<center>

Thank you!

</center>

---

### ⏸ Optional depth from here

**Today's lecture path and the notebook's close are complete.** Everything below this line is optional depth: run it if you want to push the ideas further. Nothing here is required, and any 🏠-marked prompt above is optional too.

## 8. Optional Depth: Harden the Record

**Guiding question:** *what would a stranger still need from you, and how do you
find out before they ask?*

The v1 records cover the essentials. Two exercises push them toward the
standard your replication module will demand later in the course.

First, a same-session stability check. The cell below runs the whole pipeline
twice, top to bottom, and confirms the two runs agree. This is weaker than next
week's clean restart, which clears the kernel's memory entirely, but it catches
the crudest failure early: a pipeline whose answer depends on leftover state.

Second, the delegation audit below, which turns your AI on the gap between
your code and your records.

**🏠 Optional depth.** Run this on your own if you want to go deeper.
**Before you ask:** write one sentence naming the undocumented choice you
suspect your own pipeline makes that your records do not yet mention.

> 💡 **AI Prompt:** "Here is my analysis pipeline: [paste your own code]. Return
> a table of every choice the code makes that a written declaration might not
> mention: which rows are kept or dropped, which columns are selected, how
> missing values are treated, and the exact set of cases the final number is
> computed over. Then list what a stranger would still need from me, beyond
> this code, to rerun it from scratch."
>
> **After running, verify (counters *illusion of completeness*: a tidy table can
> still omit the one undocumented choice that matters):**
> - [ ] Check the table against your code line by line. Anything it lists that
>   the code does not do was invented; strike it. Anything the code does that
>   the table missed goes to the top of your environment record.
> - [ ] Compare its "what a stranger still needs" list with the item you
>   committed above. If yours is missing, the tidy table just failed the one
>   test it looked complete on.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# OPTIONAL DEPTH — same-session stability: run the whole pipeline twice and compare.
run_a = step3_estimate(step2_transform(step1_load()))
run_b = step3_estimate(step2_transform(step1_load()))
print(f"  run A : {run_a*100:+.4f} points")
print(f"  run B : {run_b*100:+.4f} points")
print(f"  identical: {run_a == run_b}")
assert run_a == run_b, "two same-session runs disagree — the pipeline depends on leftover state"
print("✓ Same-session stability holds. The stronger test, a clean restart with the kernel's")
print("  memory cleared, is exactly what next week's M7 verification performs.")